# Data Ingestion, Cleaning & Preprocessing with Pandas

**Dataset:** Retail Store Sales: Dirty for Data Cleaning (Kaggle), 12,575 rows x 11 columns, one row per transaction.

**Goal:** turn a messy retail transactions file into a clean, standardized dataset (`clean_dataset.csv`), and show the data before and after cleaning.

**Steps:** load the raw data, profile it, fix data types, remove duplicates, handle missing values and invalid values, flag outliers, engineer features (year, month, profit margin), verify the result, export.

**How to run:** upload `retail_store_sales.csv` (Colab: Files panel), then click Runtime, Run all.

## 1. Load the raw data

In [40]:
import glob
import numpy as np
import pandas as pd

# ---- ASSUMPTIONS for profit margin ------------------------------------------
# The dataset has NO cost column, so margin cannot be measured directly.
# These two values are assumptions made for this exercise. Change them if real
# cost information becomes available.
COST_RATIO = 0.70       # assumed unit cost = 70% of price per unit
DISCOUNT_RATE = 0.10    # assumed: a transaction marked "discount applied" got 10% off
# -----------------------------------------------------------------------------

paths = (glob.glob('/content/**/retail_store_sales.csv', recursive=True)
         + glob.glob('/kaggle/input/**/retail_store_sales.csv', recursive=True)
         + glob.glob('./**/retail_store_sales.csv', recursive=True))
assert paths, 'retail_store_sales.csv not found. Upload it first, then run again.'
print('Using file:', paths[0])

raw = pd.read_csv(paths[0])
print('Shape:', raw.shape)
raw.head()

Using file: /content/retail_store_sales.csv
Shape: (12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


## 2. Profile the raw data (BEFORE cleaning)

In [41]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [42]:
# Text that really means "missing" (these are not real values)
PLACEHOLDERS = ['ERROR', 'UNKNOWN', 'N/A', 'NA', 'NULL', 'NONE', 'NAN', '']

def placeholder_mask(s):
    return s.notnull() & s.astype(str).str.strip().str.upper().isin(PLACEHOLDERS)

before = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing': raw.isnull().sum(),
    'missing_%': (raw.isnull().mean() * 100).round(1),
    'placeholder_text': [int(placeholder_mask(raw[c]).sum()) for c in raw.columns],
    'unique_values': raw.nunique(),
})
print('Duplicate rows:', int(raw.duplicated().sum()))
print('Duplicate Transaction IDs:', int(raw['Transaction ID'].duplicated().sum()))
before

Duplicate rows: 0
Duplicate Transaction IDs: 0


,dtype,missing,missing_%,placeholder_text,unique_values
Transaction ID,object,0,0.0,0,12575
Customer ID,object,0,0.0,0,25
Category,object,0,0.0,0,8
Item,object,1213,9.6,0,200
Price Per Unit,float64,609,4.8,0,25
Quantity,float64,604,4.8,0,10
Total Spent,float64,604,4.8,0,227
Payment Method,object,0,0.0,0,3
Location,object,0,0.0,0,2
Transaction Date,object,0,0.0,0,1114


In [43]:
# Numeric and date columns: do they hold text, zero/negative numbers, or bad dates?
num_cols_raw = ['Price Per Unit', 'Quantity', 'Total Spent']
nums = raw[num_cols_raw].apply(pd.to_numeric, errors='coerce')
print('Text that is not a number:', (nums.isnull() & raw[num_cols_raw].notnull()).sum().to_dict())
print('Zero or negative values:', (nums <= 0).sum().to_dict())

dates = pd.to_datetime(raw['Transaction Date'], errors='coerce')
print('Missing or unparseable dates:', int(dates.isnull().sum()))
print('Date range:', dates.min(), 'to', dates.max())
nums.describe().round(2)

Text that is not a number: {'Price Per Unit': 0, 'Quantity': 0, 'Total Spent': 0}
Zero or negative values: {'Price Per Unit': 0, 'Quantity': 0, 'Total Spent': 0}
Missing or unparseable dates: 0
Date range: 2022-01-01 00:00:00 to 2025-01-18 00:00:00


,Price Per Unit,Quantity,Total Spent
count,11966.00,11971.00,11971.00
mean,23.37,5.54,129.65
std,10.74,2.86,94.75
min,5.00,1.00,5.00
25%,14.00,3.00,51.00
50%,23.00,6.00,108.50
75%,33.50,8.00,192.00
max,41.00,10.00,410.00


## 3. Clean the data

Cleaning rules used (so every decision is explainable):
- **Placeholders** like `ERROR` or `UNKNOWN` are treated as missing.
- **Types:** numbers become numeric, dates become datetime, the discount flag becomes Yes / No / Unknown.
- **Invalid values** (zero or negative price, quantity or total) are treated as missing.
- **Duplicates:** exact duplicate rows and repeated Transaction IDs are removed (first one kept).
- **Missing values are rebuilt from the other columns when possible** (for example Total Spent = Price x Quantity), otherwise filled with a typical value or labelled `Unknown`. Rows with no usable date are dropped, because a date cannot be guessed.
- **Outliers** are flagged, not removed, because a large purchase can be genuine.

In [44]:
log = {}                      # keeps a count of every fix, shown at the end
df = raw.copy()

# 3.1 Standard column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


In [45]:
# 3.2 Placeholders -> missing, and tidy text
text_cols = df.select_dtypes(include='object').columns
n_placeholder = 0
for c in text_cols:
    s = df[c].where(df[c].isnull(), df[c].astype(str).str.strip())
    mask = placeholder_mask(s)
    n_placeholder += int(mask.sum())
    df[c] = s.mask(mask)
log['placeholder values (ERROR/UNKNOWN...) set to missing'] = n_placeholder

for c in ['category', 'payment_method', 'location']:
    df[c] = df[c].str.title()

In [46]:
# 3.3 Fix data types
for c in ['price_per_unit', 'quantity', 'total_spent']:
    before_na = int(df[c].isnull().sum())
    df[c] = pd.to_numeric(df[c], errors='coerce')
    log[f'{c}: other non-numeric text set to missing'] = int(df[c].isnull().sum()) - before_na

df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

df['discount_applied'] = (df['discount_applied'].astype(str).str.strip().str.lower()
                          .map({'true': 'Yes', 'false': 'No', 'yes': 'Yes', 'no': 'No',
                                '1': 'Yes', '0': 'No'})
                          .fillna('Unknown'))
df.dtypes

,0
transaction_id,object
customer_id,object
category,object
item,object
price_per_unit,float64
quantity,float64
total_spent,float64
payment_method,object
location,object
transaction_date,datetime64[ns]


In [47]:
# 3.4 Invalid values (zero / negative / fractional quantity) -> missing
bad_price = df['price_per_unit'] <= 0
bad_qty = (df['quantity'] <= 0) | (df['quantity'] % 1 != 0)
bad_total = df['total_spent'] <= 0
log['invalid price/quantity/total set to missing'] = int(bad_price.sum() + bad_qty.sum() + bad_total.sum())
df.loc[bad_price, 'price_per_unit'] = np.nan
df.loc[bad_qty, 'quantity'] = np.nan
df.loc[bad_total, 'total_spent'] = np.nan

In [48]:
# 3.5 Duplicates
n0 = len(df)
df = df.drop_duplicates()
log['exact duplicate rows removed'] = n0 - len(df)

n1 = len(df)
dup_id = df.duplicated('transaction_id', keep='first') & df['transaction_id'].notnull()
df = df[~dup_id]
log['repeated transaction IDs removed'] = n1 - len(df)
print('Rows now:', len(df))

Rows now: 12575


In [49]:
# 3.6 Missing values
missing_before = df.isnull().sum()

# (a) Category from the item (each item belongs to one category)
cat_by_item = df.dropna(subset=['item', 'category']).groupby('item')['category'].agg(lambda s: s.mode().iat[0])
df['category'] = df['category'].fillna(df['item'].map(cat_by_item))

# (b) Price from Total Spent / Quantity
m = df['price_per_unit'].isnull() & df['total_spent'].notnull() & df['quantity'].notnull()
df.loc[m, 'price_per_unit'] = df.loc[m, 'total_spent'] / df.loc[m, 'quantity']

# (c) Item from (category, price) when that pair belongs to exactly one item
known = df.dropna(subset=['item', 'category', 'price_per_unit'])
g = known.groupby(['category', 'price_per_unit'])['item']
unique_map = g.first()[g.nunique() == 1]
keys = pd.MultiIndex.from_frame(df[['category', 'price_per_unit']])
inferred = pd.Series(unique_map.reindex(keys).values, index=df.index)
df['item'] = df['item'].fillna(inferred).fillna('Unknown')

# (d) Price still missing: typical price of the same item, then of the same category
df['price_per_unit'] = df['price_per_unit'].fillna(df.groupby('item')['price_per_unit'].transform('median'))
df['price_per_unit'] = df['price_per_unit'].fillna(df.groupby('category')['price_per_unit'].transform('median'))

# (e) Quantity from Total Spent / Price when it comes out as a whole number
q = df['total_spent'] / df['price_per_unit']
m = df['quantity'].isnull() & q.notnull() & ((q - q.round()).abs() < 1e-6) & (q.round() > 0)
df.loc[m, 'quantity'] = q[m].round()
#     otherwise the typical (median) quantity of the category
df['quantity'] = df['quantity'].fillna(df.groupby('category')['quantity'].transform('median').round())
df['quantity'] = df['quantity'].fillna(round(df['quantity'].median()))

# (f) Total Spent = Price x Quantity
m = df['total_spent'].isnull() & df['price_per_unit'].notnull()
df.loc[m, 'total_spent'] = df.loc[m, 'price_per_unit'] * df.loc[m, 'quantity']

# (g) Labels for the remaining text columns
for c in ['customer_id', 'category', 'payment_method', 'location', 'transaction_id']:
    df[c] = df[c].fillna('Unknown')

# (h) Rows we cannot repair: no date, or no price
n2 = len(df)
df = df.dropna(subset=['transaction_date', 'price_per_unit', 'total_spent']).copy()
log['rows dropped (no date or no price)'] = n2 - len(df)

missing_after = df.isnull().sum()
fixed = pd.DataFrame({'missing_before': missing_before, 'missing_after': missing_after})
fixed

,missing_before,missing_after
transaction_id,0,0
customer_id,0,0
category,0,0
item,1213,0
price_per_unit,609,0
quantity,604,0
total_spent,604,0
payment_method,0,0
location,0,0
transaction_date,0,0


In [50]:
# 3.7 Consistency check: Total Spent must equal Price x Quantity
calc = (df['price_per_unit'] * df['quantity']).round(2)
mismatch = (df['total_spent'] - calc).abs() > 0.01
log['total_spent corrected to price x quantity'] = int(mismatch.sum())
df.loc[mismatch, 'total_spent'] = calc[mismatch]

df['quantity'] = df['quantity'].astype(int)

In [51]:
# 3.8 Outliers (IQR rule): flagged, not removed
def iqr_flag(s):
    q1, q3 = s.quantile([0.25, 0.75])
    i = q3 - q1
    return (s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)

df['is_outlier'] = iqr_flag(df['total_spent']) | iqr_flag(df['quantity']) | iqr_flag(df['price_per_unit'])
log['rows flagged as outliers (kept)'] = int(df['is_outlier'].sum())
df[['price_per_unit', 'quantity', 'total_spent']].describe().round(2)

,price_per_unit,quantity,total_spent
count,12575.00,12575.00,12575.00
mean,23.37,5.55,130.08
std,10.75,2.79,93.54
min,5.00,1.00,5.00
25%,14.00,3.00,52.00
50%,23.00,6.00,110.00
75%,33.50,8.00,192.00
max,41.00,10.00,410.00


## 4. Feature engineering

- **Date features:** year, month, month name, quarter, day of week, weekend flag.
- **Profit margin:** the dataset has no cost column, so this uses the two assumptions set at the top of the notebook (`COST_RATIO`, `DISCOUNT_RATE`). Treat the margin as an estimate that shows how the pieces fit together, not as real profit.

  net revenue = Total Spent, reduced by `DISCOUNT_RATE` when a discount was applied
  estimated cost = Price x Quantity x `COST_RATIO`
  profit margin % = (net revenue - estimated cost) / net revenue x 100

In [52]:
dt = df['transaction_date'].dt
df['year'] = dt.year
df['month'] = dt.month
df['month_name'] = dt.month_name()
df['quarter'] = dt.quarter
df['day_of_week'] = dt.day_name()
df['is_weekend'] = dt.dayofweek >= 5

df['net_revenue'] = np.where(df['discount_applied'] == 'Yes',
                             df['total_spent'] * (1 - DISCOUNT_RATE),
                             df['total_spent']).round(2)
df['estimated_cost'] = (df['price_per_unit'] * df['quantity'] * COST_RATIO).round(2)
df['estimated_profit'] = (df['net_revenue'] - df['estimated_cost']).round(2)
df['profit_margin_pct'] = (df['estimated_profit'] / df['net_revenue'] * 100).round(2)

df[['transaction_date', 'year', 'month', 'month_name', 'quarter', 'day_of_week',
    'total_spent', 'net_revenue', 'estimated_cost', 'estimated_profit', 'profit_margin_pct']].head()

,transaction_date,year,month,month_name,quarter,day_of_week,total_spent,net_revenue,estimated_cost,estimated_profit,profit_margin_pct
0,2024-04-08,2024,4,April,2,Monday,185.0,166.5,129.50,37.00,22.22
1,2023-07-23,2023,7,July,3,Sunday,261.0,234.9,182.70,52.20,22.22
2,2022-10-05,2022,10,October,4,Wednesday,43.0,43.0,30.10,12.90,30.00
3,2022-05-07,2022,5,May,2,Saturday,247.5,247.5,173.25,74.25,30.00
4,2022-10-02,2022,10,October,4,Sunday,87.5,87.5,61.25,26.25,30.00


## 5. Verify the clean data

In [53]:
df = df.reset_index(drop=True)

checks = {
    'no duplicate rows': int(df.duplicated().sum()) == 0,
    'no duplicate transaction IDs (excluding Unknown)':
        int(df[df['transaction_id'] != 'Unknown']['transaction_id'].duplicated().sum()) == 0,
    'no missing values': int(df.isnull().sum().sum()) == 0,
    'dates are datetime': str(df['transaction_date'].dtype).startswith('datetime64'),
    'price, quantity, total are positive': bool(((df['price_per_unit'] > 0) & (df['quantity'] > 0) & (df['total_spent'] > 0)).all()),
    'total_spent = price x quantity': bool(((df['total_spent'] - df['price_per_unit'] * df['quantity']).abs() < 0.01).all()),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)

PASS - no duplicate rows
PASS - no duplicate transaction IDs (excluding Unknown)
PASS - no missing values
PASS - dates are datetime
PASS - price, quantity, total are positive
PASS - total_spent = price x quantity


## 6. Before vs after

In [54]:
summary = pd.DataFrame({
    'before (raw)': [raw.shape[0], raw.shape[1], int(raw.isnull().sum().sum()),
                     int(raw.duplicated().sum()),
                     int(raw['Transaction ID'].duplicated().sum())],
    'after (clean)': [df.shape[0], df.shape[1], int(df.isnull().sum().sum()),
                      int(df.duplicated().sum()),
                      int(df['transaction_id'].duplicated().sum())],
}, index=['rows', 'columns', 'missing cells', 'duplicate rows', 'duplicate transaction IDs'])
display(summary)

print('Data types, before and after:')
display(pd.DataFrame({'before': raw.dtypes.astype(str).rename(lambda c: c.strip().lower().replace(' ', '_')),
                      'after': df.dtypes.astype(str)}).fillna('(new column)'))

print('What the cleaning changed:')
display(pd.Series(log, name='count').to_frame())

,before (raw),after (clean)
rows,12575,12575
columns,11,22
missing cells,7229,0
duplicate rows,0,0
duplicate transaction IDs,0,0


Data types, before and after:


,before,after
category,object,object
customer_id,object,object
day_of_week,(new column),object
discount_applied,object,object
estimated_cost,(new column),float64
estimated_profit,(new column),float64
is_outlier,(new column),bool
is_weekend,(new column),bool
item,object,object
location,object,object


What the cleaning changed:


,count
placeholder values (ERROR/UNKNOWN...) set to missing,0
price_per_unit: other non-numeric text set to missing,0
quantity: other non-numeric text set to missing,0
total_spent: other non-numeric text set to missing,0
invalid price/quantity/total set to missing,604
exact duplicate rows removed,0
repeated transaction IDs removed,0
rows dropped (no date or no price),0
total_spent corrected to price x quantity,0
rows flagged as outliers (kept),60


In [55]:
print('RAW (first rows):')
display(raw.head())
print('CLEAN (first rows):')
display(df.head())

RAW (first rows):


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


CLEAN (first rows):


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,...,year,month,month_name,quarter,day_of_week,is_weekend,net_revenue,estimated_cost,estimated_profit,profit_margin_pct
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,2024-04-08,...,2024,4,April,2,Monday,False,166.5,129.50,37.00,22.22
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,2023-07-23,...,2023,7,July,3,Sunday,True,234.9,182.70,52.20,22.22
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,2022-10-05,...,2022,10,October,4,Wednesday,False,43.0,30.10,12.90,30.00
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,2022-05-07,...,2022,5,May,2,Saturday,True,247.5,173.25,74.25,30.00
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,2022-10-02,...,2022,10,October,4,Sunday,True,87.5,61.25,26.25,30.00


## 7. Export the clean dataset

In [56]:
df.to_csv('clean_dataset.csv', index=False)
print('Saved clean_dataset.csv with', df.shape[0], 'rows and', df.shape[1], 'columns')

try:
    from google.colab import files
    files.download('clean_dataset.csv')      # downloads it to your computer
except Exception:
    pass

Saved clean_dataset.csv with 12575 rows and 22 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes and limitations

- Profit margin is an **estimate based on assumed cost and discount rates**, because the source data has no cost column.
- Repaired values (price, quantity, total) were derived from other columns of the same row, or from typical values of the same item or category.
- Rows with no usable transaction date or price were dropped; the count is in the cleaning log above.
- Outliers are flagged in `is_outlier` and kept.